# Klasifikasi DemogPairs Menggunakan ViT (Emosi) & Gaussian Naive Bayes

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-emotion.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
var_smoothing_values = np.logspace(-9, 2, 40)  # dari 1e-9 sampai 1e2, 40 nilai

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [GaussianNB()],
        'classifier__var_smoothing': var_smoothing_values
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

GaussianNB: 240 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_gnb_vit-emotion_',
    results_path='results/demogpairs_gnb_vit-emotion_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: GaussianNB


{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.7337962962962963
Precision : 0.7386856751531776
Recall    : 0.7337962962962964
F1 Score  : 0.7329054843249007
               precision    recall  f1-score   support

Asian_Females     0.7659    0.6361    0.6950       360
  Asian_Males     0.6694    0.6806    0.6749       360
Black_Females     0.7410    0.7472    0.7441       360
  Black_Males     0.8306    0.6944    0.7564       360
White_Females     0.6929    0.8083    0.7462       360
  White_Males     0.7324    0.8361    0.7808       360

     accuracy                         0.7338      2160
    macro avg     0.7387    0.7338    0.7329      2160
 weighted avg     0.7387    0.7338    0.7329      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9069444444444444,0.7658862876254181,0.6361111111111111,0.6949924127465857,360
Asian_Males,0.8907407407407407,0.6693989071038251,0.6805555555555556,0.674931129476584,360
Black_Females,0.9143518518518519,0.7410468319559229,0.7472222222222222,0.7441217150760719,360
Black_Males,0.9254629629629629,0.8305647840531561,0.6944444444444444,0.75642965204236,360
White_Females,0.9083333333333333,0.6928571428571428,0.8083333333333333,0.7461538461538462,360
White_Males,0.9217592592592593,0.732360097323601,0.8361111111111111,0.7808041504539559,360


Confusion matrix saved: images\cm_gnb_vit-emotion_GaussianNB.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               229                44                24                 1                59                 3
         Asian_Males                32               245                 7                22                 8                46
       Black_Females                14                12               269                23                37                 5
         Black_Males                 0                28                43               250                 4                35
       White_Females                23                 6                19                 0               291                21
         White_Males                 1                31                 1                 5                21               301


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
GaussianNB,models/clf_demogpairs_gnb_vit-emotion_GaussianNB.pkl,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': None}",0.7337962962962963,0.7329054843249007,0.7386856751531776,0.7337962962962964,240


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_gnb_vit-emotion_GaussianNB.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 476.0,
 'days': 0,
 'hours': 0,
 'minutes': 7,
 'seconds': 56.0,
 'text': '0 hari 0 jam 7 menit 56.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 2287.0,
 'days': 0,
 'hours': 0,
 'minutes': 38,
 'seconds': 7.0,
 'text': '0 hari 0 jam 38 menit 7.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00307029062975785), 'pca': 'PCA', 'scaler': None}",0.7222,0.7355,0.7245,0.7251,0.7483,0.7311,0.7306,0.738,0.7311,1.948
2,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.0058780160722749115), 'pca': 'PCA', 'scaler': None}",0.7251,0.735,0.7257,0.7199,0.7471,0.7306,0.7301,0.7379,0.7306,1.9386
3,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.001603718743751331), 'pca': 'PCA', 'scaler': None}",0.7193,0.7344,0.7263,0.7251,0.7471,0.7304,0.73,0.7369,0.7304,2.0376
4,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(0.00022854638641349884), 'pca': 'PCA', 'scaler': None}",0.7188,0.7373,0.724,0.7263,0.7454,0.7303,0.7299,0.7366,0.7303,1.4449
...,...,...,...,...,...,...,...,...,...,...,...
237,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.4902,0.4722,0.4884,0.5,0.5041,0.491,0.4826,0.637,0.491,3.4377
238,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(27.283333764867695), 'pca': 'PCA', 'scaler': None}",0.4844,0.4722,0.4838,0.4873,0.5029,0.4861,0.4769,0.6355,0.4861,1.2702
239,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(52.233450742668325), 'pca': 'PCA', 'scaler': None}",0.4821,0.4693,0.478,0.4878,0.5,0.4834,0.4731,0.6333,0.4834,1.8656
240,"{'classifier': 'GaussianNB', 'classifier__var_smoothing': np.float64(100.0), 'pca': 'PCA', 'scaler': None}",0.4826,0.4682,0.4786,0.4873,0.4971,0.4828,0.4722,0.6336,0.4828,1.6719
